In [1]:
!pip install -q google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 5.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-cloud-translate 3.12.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 5.29.5 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 46.0.3 which is incompatible.
pydrive2 1.21.3 requires pyOpenSSL<=24.2.1,>=19.1.0, but you have pyopenssl 25.3.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.10.0 wh

In [2]:
import os
import re
import time
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import classification_report, accuracy_score

import google.generativeai as genai
from kaggle_secrets import UserSecretsClient

In [3]:
try:
    user_secrets = UserSecretsClient()
    GEMINI_API_KEY = user_secrets.get_secret("GEMINI_API_KEY")
except:
    raise RuntimeError("GEMINI_API_KEY not found")

genai.configure(api_key=GEMINI_API_KEY)

In [4]:
class Config:
    MODEL_NAME = "gemini-2.5-flash"

    POS_DATA_FILE = "/kaggle/input/lrec-tcs-attack-support/sentencePair.txt"
    NEG_DATA_FILE = "/kaggle/input/lrec-tcs-attack-support/sentencePair_neg.txt"
    OUTPUT_FILE = "/kaggle/working/gemini_fewshot_cot.csv"

    SAMPLE_SIZE = 500
    RANDOM_SEED = 42

    TEMPERATURE = 0.0
    MAX_OUTPUT_TOKENS = 2048
    REQUEST_SLEEP = 0.2

In [5]:
# =============================================================================
#  DATA LOADING & PREPARATION
# =============================================================================
FILENAME_PATTERN = re.compile(r"SupremeCourt_(\d{4})_(\d+)\.txt")

def parse_lrec_line(line: str):
    parts = line.strip().split("\t")
    fname_indices = [i for i, p in enumerate(parts) if p.endswith(".txt")]
    if len(fname_indices) != 2: return None
    try:
        sentpair_id = int(parts[0])
        sent1 = " ".join(parts[fname_indices[0] + 2 : fname_indices[1]]).strip('"')
        sent2 = " ".join(parts[fname_indices[1] + 2 : len(parts) - 2]).strip('"')
        label = parts[-1]
    except (ValueError, IndexError): return None
    return {"id": sentpair_id, "sent1": sent1, "sent2": sent2, "label": label}

def load_lrec_file(filepath: str) -> pd.DataFrame:
    rows = []
    print(f"Reading data from {filepath}...")
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            if parsed_row := parse_lrec_line(line):
                rows.append(parsed_row)
    return pd.DataFrame(rows)

def load_and_split_data(config: Config):
    """
    Returns:
      df_test: test dataframe
      df_shots: pool from which few-shot exemplars are drawn (disjoint from df_test)
    """
    print("Loading and splitting data...")
    df_pos = load_lrec_file(config.POS_DATA_FILE)
    df_neg = load_lrec_file(config.NEG_DATA_FILE)
    if df_pos.empty or df_neg.empty:
        print("ERROR: Data files failed to load.")
        return None, None

    # Few-shot pool (sample a chunk to keep memory & speed reasonable)
    df_pos_pool = df_pos.sample(n=min(10000, len(df_pos)), random_state=config.RANDOM_SEED)
    df_neg_pool = df_neg.sample(n=min(10000, len(df_neg)), random_state=config.RANDOM_SEED)
    df_shots = pd.concat([df_pos_pool, df_neg_pool]).reset_index(drop=True)

    # Test set excludes the shot pool to avoid leakage
    remaining_pos = df_pos.drop(df_pos_pool.index)
    remaining_neg = df_neg.drop(df_neg_pool.index)
    df_test = pd.concat([remaining_pos, remaining_neg]).reset_index(drop=True)
    df_test = df_test.dropna(subset=['id', 'sent1', 'sent2', 'label'])
    print(f"Created test set with {len(df_test)} samples.")

    if config.SAMPLE_SIZE > 0:
        print(f"Using a sample of {config.SAMPLE_SIZE} examples from the test set for evaluation.")
        df_test = df_test.sample(n=min(config.SAMPLE_SIZE, len(df_test)), random_state=config.RANDOM_SEED).reset_index(drop=True)

    return df_test, df_shots

In [6]:
FEWSHOT_COT_PROMPT = """
You are an expert legal analyst specializing in argumentation theory.

Your task is to classify the relationship between two sentences from Indian Supreme Court judgments.

Possible labels:
- SUPPORT: Sentence 2 strengthens or logically supports Sentence 1
- ATTACK: Sentence 2 weakens, rebuts, limits, or challenges Sentence 1
- NO_REL: The sentences are unrelated

For each example:
1. Give a short legal reasoning (2–3 sentences)
2. Output the final label in the exact format:
   Final Label: <SUPPORT|ATTACK|NO_REL>

---
Example 1:
Sentence 1: "The Tribunal misdirected itself in law."
Sentence 2: "The appeal is therefore allowed and remanded."
Reasoning: The second sentence states the legal consequence of the error identified in the first sentence. It follows directly from the finding of misdirection.
Final Label: SUPPORT

---
Example 2:
Sentence 1: "The conviction was based on unlawful assembly."
Sentence 2: "The accused was acquitted under Section 148 IPC."
Reasoning: The second sentence negates the factual basis required for the conviction mentioned in the first sentence. This undermines the original legal conclusion.
Final Label: ATTACK

---
Example 3:
Sentence 1: "The defendant was guilty of perjury."
Sentence 2: "The Companies Act amendment comes into force next month."
Reasoning: The two sentences concern unrelated legal topics and do not affect each other.
Final Label: NO_REL

---
Now analyze the following pair.

Sentence 1: "{sent1}"
Sentence 2: "{sent2}"

Reasoning:
"""

In [7]:
def extract_text_and_tokens(response):
    """
    Returns:
      text: str or None
      usage: dict with token counts
      finish_reason: str
    """
    text = None
    finish_reason = None

    if response.candidates:
        cand = response.candidates[0]
        finish_reason = cand.finish_reason

        if cand.content and cand.content.parts:
            part = cand.content.parts[0]
            if hasattr(part, "text"):
                text = part.text

    usage = {}
    if hasattr(response, "usage_metadata"):
        usage = {
            "prompt_tokens": response.usage_metadata.prompt_token_count,
            "output_tokens": response.usage_metadata.candidates_token_count,
            "total_tokens": response.usage_metadata.total_token_count,
        }

    return text, usage, finish_reason

In [8]:
class GeminiFewShotCoTClassifier:
    def __init__(self, config: Config):
        self.config = config
        self.model = genai.GenerativeModel(config.MODEL_NAME)

    def parse_output(self, text: str) -> str:
        text = text.upper()
        if "FINAL LABEL:" in text:
            label = text.split("FINAL LABEL:")[-1].strip().split()[0]
            if label in {"SUPPORT", "ATTACK", "NO_REL"}:
                return label
        return "NO_REL"  # safe fallback

    def predict(self, df: pd.DataFrame):
        predictions = []
        total_token = []
        output_token = []
        prompt_token = []
        reason = [] 

        for _, row in tqdm(df.iterrows(), total=len(df)):
            prompt = FEWSHOT_COT_PROMPT.format(
                sent1=row.sent1,
                sent2=row.sent2
            )

            try:
                response = self.model.generate_content(
                    prompt,
                    generation_config={
                        "temperature": self.config.TEMPERATURE,
                        "max_output_tokens": self.config.MAX_OUTPUT_TOKENS
                    }
                )
                
                text1, usage1, finish_reason1 = extract_text_and_tokens(response)
                temp_total_token = usage1['total_tokens']
                temp_output_token = usage1['output_tokens']
                temp_prompt_token = usage1['prompt_tokens']
                pred = self.parse_output(response.text)
            except Exception as e:
                print("The error in FewShot CoT is ", e)
                pred = "NO_REL"
                temp_total_token = 0
                temp_output_token = 0
                temp_prompt_token = 0
                text1 = ""

            predictions.append(pred)
            total_token.append(temp_total_token)
            output_token.append(temp_output_token)
            prompt_token.append(temp_prompt_token)
            reason.append(text1)
            time.sleep(self.config.REQUEST_SLEEP)

        return predictions, total_token, output_token, prompt_token, reason

In [9]:
if __name__ == "__main__":
    config = Config()
    df_test,df_shots = load_and_split_data(config)

    clf = GeminiFewShotCoTClassifier(config)
    preds, total_token, output_token, prompt_token, reason = clf.predict(df_test)

    true_labels = (
        df_test["label"]
        .str.upper()
        .str.replace(" ", "_")
        .replace({"NO_RELATION": "NO_REL"})
    )

    print("Accuracy:", accuracy_score(true_labels, preds))
    print(classification_report(true_labels, preds, zero_division=0))

    out = pd.DataFrame({
        "pair_id": df_test["id"],
        "sentence_1": df_test["sent1"],
        "sentence_2": df_test["sent2"],
        "true_label": df_test["label"],
        "predicted_label": preds,
        "prompt_token": prompt_token,
        "output_token": output_token,
        "total_token": total_token,
        "reason": reason
    })

    out.to_csv(config.OUTPUT_FILE, index=False)

Loading and splitting data...
Reading data from /kaggle/input/lrec-tcs-attack-support/sentencePair.txt...
Reading data from /kaggle/input/lrec-tcs-attack-support/sentencePair_neg.txt...
Created test set with 20506 samples.
Using a sample of 500 examples from the test set for evaluation.


100%|██████████| 500/500 [45:51<00:00,  5.50s/it]

Accuracy: 0.488
              precision    recall  f1-score   support

      ATTACK       0.43      0.38      0.41       130
      NO_REL       0.76      0.42      0.54       247
     SUPPORT       0.36      0.73      0.49       123

    accuracy                           0.49       500
   macro avg       0.52      0.51      0.48       500
weighted avg       0.58      0.49      0.49       500

